# Ruteo de ambulancias en CABA como problema de búsqueda

Dada una ambulancia y un incidente, se busca la ruta de **menor tiempo** ambulancia → incidente → hospital de guardia,
sobre la red vial de la Ciudad de Buenos Aires (OpenStreetMap), con los 14 hospitales generales de agudos como posibles destinos.
Se comparan **BFS, UCS, A\* (heurística h2), A\* ponderado y greedy**.

**Cómo ejecutarlo:** Ejecutar todo. La primera vez descarga el mapa de OpenStreetMap (puede tardar varios minutos).

## 1. Preparación

In [ ]:
import os, sys, subprocess

REPO_URL = "https://github.com/Agustin327/Trabajo-Integrador-AA.git"   # <- URL del repositorio
FUENTE = "osm"                                           # 

# 
if "google.colab" in sys.modules and not os.path.isdir("src"):
    nombre = REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
    if not os.path.isdir(nombre):
        subprocess.run(["git", "clone", REPO_URL], check=True)
    os.chdir(nombre)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, "src")

try:
    import osmnx  # 
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "osmnx"], check=True)

print("Directorio de trabajo:", os.getcwd())
if FUENTE == "osm" and not os.path.exists("data/hospitales.geojson"):
    print("ATENCIÓN: falta data/hospitales.geojson (ver data/README.md).")

## 2. Pruebas automáticas
Verifican, sobre un grafo sintético, que A\* con h2 da el mismo costo que UCS, que h2 es admisible y consistente,
y que el filtro deja exactamente los 14 hospitales.

In [ ]:
r = subprocess.run([sys.executable, "src/pruebas.py"], capture_output=True, text=True)
print(r.stdout, r.stderr[-2000:] if r.returncode else "")
assert r.returncode == 0, "Fallaron las pruebas"

## 3. Experimentos
Casos generados al azar con **semilla fija (42)**: la ambulancia parte de un hospital y el incidente cae en una de tres
categorías según la distancia por calle de la fase 1 (corto ≤ 3 km, medio 3–8 km, largo > 8 km).
Velocidad: 46 km/h (ver supuestos en el README).

In [ ]:
N_POR_CATEGORIA = 20
args = [sys.executable, "src/experiments.py", "--fuente", FUENTE, "--n", str(N_POR_CATEGORIA)]
if FUENTE == "sintetico":
    args += ["--corto-max", "1500", "--largo-min", "3500"]
r = subprocess.run(args, capture_output=True, text=True)
print(r.stdout, r.stderr[-3000:] if r.returncode else "")
assert r.returncode == 0, "Falló el experimento (ver mensaje arriba)"

### Resultados por categoría y algoritmo

In [ ]:
import pandas as pd
res = pd.read_csv("resultados/resumen.csv")
res["categoria"] = pd.Categorical(res["categoria"], ["corto", "medio", "largo"], ordered=True)
res = res.sort_values("categoria", kind="stable")
res

### Esfuerzo de búsqueda
Nodos expandidos promedio por algoritmo (escala logarítmica). Menos es mejor, pero solo UCS y A\* con h2 garantizan la ruta óptima.

In [ ]:
import matplotlib.pyplot as plt

pivot = res.pivot(index="categoria", columns="algo", values="expandidos").loc[["corto", "medio", "largo"]]
orden = [a for a in ["bfs", "ucs", "astar_h2", "astar_h3", "wastar_h2_1.5", "greedy_h2"] if a in pivot.columns]
ax = pivot[orden].plot(kind="bar", figsize=(9, 5), logy=True, rot=0)
ax.set_xlabel("Distancia del caso")
ax.set_ylabel("Nodos expandidos (promedio, escala log)")
ax.set_title("Esfuerzo de búsqueda por algoritmo")
ax.legend(title="Algoritmo")
plt.tight_layout()
plt.savefig("resultados/nodos_expandidos.png", dpi=150)
plt.show()

## 4. Ruta óptima con posiciones cargadas a mano
Completá `ORIGEN` e `INCIDENTE`. Cada uno acepta **coordenadas** `"lat, lon"` (por ejemplo copiadas de Google Maps),
un **hospital** `"hospital:clave"` (alvarez, argerich, durand, fernandez, grierson, penna, pinero, pirovano, ramos mejia,
rivadavia, santojanni, tornu, velez sarsfield, zubizarreta) o una **dirección**.
Con `COMPARAR = True` también corre los demás algoritmos sobre el mismo caso.

In [ ]:
from ruta import ejecutar
from IPython.display import Image, display

if FUENTE == "osm":
    ORIGEN, INCIDENTE = "hospital:pirovano", "-34.6037, -58.3816"
else:  # grafo sintético: coordenadas dentro del grafo de prueba
    ORIGEN, INCIDENTE = "hospital:h2", "-34.5800, -58.3750"

info = ejecutar(ORIGEN, INCIDENTE, fuente=FUENTE, comparar=True, salida="ruta_manual")
display(Image("ruta_manual/mapa.png"))

## 5. Notas
- El tiempo es solo de **manejo** a velocidad constante: no incluye atención del llamado, despacho ni salida de la base.
- Como la velocidad es constante, el camino óptimo es el más corto en metros; `v` solo escala los tiempos.
- Red vial © colaboradores de OpenStreetMap (ODbL). Hospitales: portal de datos abiertos del Gobierno de la Ciudad de Buenos Aires.